In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio

In [ ]:
DIR_DATA = Path("data")
DIR_METADATA = DIR_DATA / "0_metadata"
DIR_ICEYE_ALIGNED = DIR_DATA / "1-5_intermediate" / "1_ICEYE_aligned"
DIR_RESULTS = DIR_DATA / "3_results"
FILEPATH_POI = DIR_METADATA / "poi.csv"
FILEPATH_ICEYE_LIST = DIR_METADATA / "iceye-list.csv"
FILEPATH_PRECIP = DIR_METADATA / "precip.csv"
DIR_OUT = DIR_RESULTS / "sar-bs-temp-trend"

In [ ]:
poi_df = pd.read_csv(FILEPATH_POI)
sar_df = pd.read_csv(FILEPATH_ICEYE_LIST)
precip_df = pd.read_csv(FILEPATH_PRECIP)

In [ ]:
records = []

for _, sar in sar_df.iterrows():
    sar_path = DIR_ICEYE_ALIGNED / sar["filename"]
    sar_path = DIR_ICEYE_ALIGNED / (
        sar_path.stem + "_EPSG2958_res05m.tif"
    )
    print(sar_path)

    with rasterio.open(sar_path) as src:
        image = src.read(1)
        for _, point in poi_df.iterrows():
            row, col = src.index(point["x"], point["y"])

            if 0 <= row < src.height and 0 <= col < src.width:
                value = image[row, col]
            else:
                value = np.nan

            records.append({
                "sar_id": sar["dirname"],
                "date": pd.to_datetime(
                    sar["date"],
                    format="%Y/%m/%d"
                ),
                "orbit_direction": sar["orbit_direction"],
                "look_side": sar["look_side"],
                "incidence_angle": sar["incident_angle"],
                "poi_id": point["id"],
                "area" : point["area"],
                "x": point["x"],
                "y": point["y"],
                "backscatter": value
            })

backscatter_df = pd.DataFrame(records)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

precip_df["date"] = pd.to_datetime(
    precip_df["date"],
    format="%Y/%m/%d"
)

# Wet days
wet_days = precip_df[
    precip_df["Total Precip (mm)"] > 10
].copy()


backscatter_df["date"] = pd.to_datetime(backscatter_df["date"])
backscatter_df["year"] = backscatter_df["date"].dt.year

for area, area_df in backscatter_df.groupby("area"):

    fig, axes = plt.subplots(
        1, 2,
        figsize=(18, 6),
        sharey=True
    )

    for ax, year in zip(axes, [2024, 2025]):

        year_df = area_df[
            area_df["year"] == year
        ].copy()

        if year_df.empty:
            ax.set_visible(False)
            continue

        # --------------------------------------------------
        # Wet-day background bands
        # --------------------------------------------------
        year_wet_days = wet_days[
            wet_days["date"].dt.year == year
        ]

        for _, wet_day in year_wet_days.iterrows():

            start = wet_day["date"]
            end = start + pd.Timedelta(days=1)

            ax.axvspan(
                start,
                end,
                alpha=0.2,
                label="Wet day (>10 mm)"
            )

        # --------------------------------------------------
        # Individual points
        # --------------------------------------------------
        for poi_id, point_df in year_df.groupby("poi_id"):

            point_df = point_df.sort_values("date")

            ax.plot(
                point_df["date"],
                point_df["backscatter"],
                marker="o",
                linewidth=1,
                alpha=0.5,
                label=poi_id
            )

        # --------------------------------------------------
        # Mean, minimum, maximum
        # --------------------------------------------------
        summary_df = (
            year_df
            .groupby("date")["backscatter"]
            .agg(["mean", "min", "max"])
            .reset_index()
            .sort_values("date")
        )

        # Point range
        ax.fill_between(
            summary_df["date"],
            summary_df["min"],
            summary_df["max"],
            alpha=0.15,
            label="Point range"
        )

        # Mean
        ax.plot(
            summary_df["date"],
            summary_df["mean"],
            marker="o",
            linewidth=3,
            label="Mean"
        )

        # --------------------------------------------------
        # Formatting
        # --------------------------------------------------
        ax.set_title(str(year))
        ax.set_xlabel("Date")
        ax.grid(True, alpha=0.3)

        # Show acquisition dates
        dates = summary_df["date"].tolist()

        ax.set_xticks(dates)

        ax.xaxis.set_major_formatter(
            mdates.DateFormatter("%b %d")
        )

        plt.setp(
            ax.get_xticklabels(),
            rotation=45,
            ha="right"
        )

    axes[0].set_ylabel("Backscatter")

    # ------------------------------------------------------
    # Legend
    # ------------------------------------------------------
    handles, labels = axes[0].get_legend_handles_labels()

    # Remove duplicate labels
    unique = dict(zip(labels, handles))

    axes[1].legend(
        unique.values(),
        unique.keys(),
        bbox_to_anchor=(1.02, 1),
        loc="upper left"
    )

    fig.suptitle(
        f"SAR Backscatter Temporal Trend — {area}",
        fontsize=14
    )

    plt.tight_layout()

    # Save figure
    output_path = DIR_OUT / f"bstrend_{area}.png"
    fig.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)
